# 02 - Silver: RUES (Registro Mercantil)

Trabajo Práctico 2 - Transformación Bronze -> Silver con reglas de calidad (DQL)

## Objetivo

Tipar y validar `Datos_Empresas.bronze.DE_Semiestructurado_RegistroMercantil_Api` (2.815.313 filas). Las filas que pasan todas las reglas de calidad van a `Datos_Empresas.silver.empresas_rues`; las que incumplen al menos una van a `Datos_Empresas.dql.rechazados_rues` con el motivo del rechazo.

## Transformaciones de tipado

* `fecha_matricula`, `fecha_renovacion`, `fecha_cancelacion`: texto `YYYYMMDD` -> `DATE` (si no tienen 8 dígitos o no son una fecha de calendario válida, quedan `NULL`).
* `fecha_vigencia`: igual, pero además el valor centinela `99991231` ("sin vencimiento") se convierte explícitamente en `NULL`.
* `fecha_actualizacion`: texto `YYYY/MM/DD HH:MM:SS.fffffffff` -> `TIMESTAMP`.
* `ultimo_ano_renovado`: texto -> `INT`.
* `matricula` y `numero_identificacion`: se recortan espacios (`trim`) antes de validar.
* El resto de columnas se conserva como texto (son identificadores o categorías, no tiene sentido tipar como número).

## Reglas de calidad (rechazo)

| # | Regla | Justificación |
|---|---|---|
| 1 | `matricula` o `numero_identificacion` vacíos, **o `numero_identificacion` es el centinela `"0000000000000"`** | Son las columnas que identifican a la empresa; sin ellas el registro no se puede usar ni enlazar en Gold. El centinela se descubrió al construir `gold.fact_contratos`: 7.338 empresas completamente distintas (razón social, matrícula, cámara distintos) comparten ese mismo valor — es un placeholder de "identificación no capturada" de la fuente, no un identificador real. Si se deja pasar, cualquier join por `numero_identificacion` (como el de Gold con SECOP) mezclaría 7.338 empresas no relacionadas entre sí. |
| 2 | Combinación (`codigo_camara`, `matricula`) duplicada — **solo las copias antiguas** | Regla de unicidad: una empresa no puede tener más de un registro activo con la misma matrícula en la misma cámara. Cuando hay duplicados se conserva en Silver la copia más reciente (mayor `fecha_actualizacion`) y se rechazan únicamente las demás — no se descarta la matrícula completa. Verificado en vivo: de 309.418 filas en grupos duplicados, 154.709 son la versión más reciente (se conservan) y 154.709 son copias antiguas (se rechazan). |
| 3 | `fecha_matricula` con formato/calendario inválido, en el futuro, o **anterior a 1900** | Una fecha de matrícula mal formada o futura es imposible en la realidad del negocio — ya se detectaba como outlier en el TP1. Se agregó el piso de 1900 tras construir la capa Gold: se encontraron 7 filas en Bronze con el centinela `'00010101'`/`'10000101'` (año 1 o 1000), que `try_to_date` acepta como fecha de calendario válida pero que generaban un "mes fantasma" en `gold.fact_rues_mensual` — no son fechas reales de matrícula, son placeholders de dato faltante. |
| 4 | `cod_ciiu_act_econ_pri` vacío | Sin código de actividad económica principal no se puede clasificar la empresa por sector, que es justo lo que necesitamos para la pregunta de negocio "actividades económicas más frecuentes por región" en Gold. Se verificó en vivo contra Bronze: afecta solo 31 de 2.815.313 filas (~0.001%), y el 100% de esas 31 son empresas con `estado_matricula = CANCELADA` o `MATRÍCULA CANCELADA LEY 1429` — el dato ya llega vacío desde la fuente para registros cancelados, no es un error de nuestra transformación. |

Un registro puede incumplir varias reglas a la vez; todas quedan registradas en `_motivo_rechazo`, separadas por `; `.

**Nota sobre `numero_identificacion` no único:** incluso después de quitar el centinela, 57.108 identificaciones distintas aparecen en más de una fila de `silver.empresas_rues` (empresas con matrícula en más de una cámara de comercio). Esto es legítimo — **no** es una regla de rechazo — pero significa que `numero_identificacion` no es clave única en Silver; cualquier join por esa columna (como en `gold.fact_contratos`) debe deduplicar explícitamente antes de unir, o generará filas duplicadas (fan-out).

In [ ]:
%python
from pyspark.sql import functions as F
from pyspark.sql.window import Window

TABLA_BRONZE = "Datos_Empresas.bronze.DE_Semiestructurado_RegistroMercantil_Api"
TABLA_SILVER = "Datos_Empresas.silver.empresas_rues"
TABLA_DQL = "Datos_Empresas.dql.rechazados_rues"

df_bronze = spark.table(TABLA_BRONZE)
print(f"Bronze: {df_bronze.count():,} filas")

---

## Paso 0: Análisis de nulos — ¿qué columnas sirven para Silver y por qué?

Antes de tipar y aplicar reglas se revisó, columna por columna (36 columnas de negocio), el % de valores nulos/vacíos contra Bronze en vivo. Resultado: **ninguna columna se elimina** — cada columna con nulos altos tiene una explicación de negocio verificable, no es un defecto de calidad:

| Columna | % vacío | Por qué (verificado contra Bronze) |
|---|---|---|
| `fecha_cancelacion` | 83.5% | Solo las empresas canceladas tienen esta fecha; el 83.5% siguen activas. |
| `ciiu4` | 83.6% | Cuarta actividad económica adicional — la mayoría de empresas solo registra 1 o 2 actividades. |
| `clase_identificacion_rl`, `num_identificacion_representante_legal`, `representante_legal` | 65.4% | Solo aplica a Persona Jurídica; una Persona Natural no tiene representante legal distinto de sí misma. |
| `ciiu3` | 68.1% | Tercera actividad económica adicional (mismo patrón que `ciiu4`). Se verificó que `ciiu3` referencia el mismo catálogo CIIU que `cod_ciiu_act_econ_pri` (497 de 499 códigos del catálogo tienen coincidencia) — **no es una codificación legacy descartable**, es un slot adicional real. |
| `sigla` | 92.8% | Una sigla (nombre corto/acrónimo) es opcional; la mayoría de empresas no registra una. |
| `fecha_vigencia` | 50.2% | Parcialmente ligado a cancelación, pero el 47% de las empresas **ACTIVAS** también tiene este campo vacío — es una inconsistencia real de la fuente RUES, no explicable 100% por reglas de negocio. No se usa en ninguna pregunta de negocio de Gold, así que se conserva en Silver tal cual (pass-through) sin convertirla en regla de rechazo. |
| `segundo_nombre` | 49.4% | No toda Persona Natural tiene segundo nombre. |
| `cod_ciiu_act_econ_sec` | 40.6% | Segunda actividad económica — opcional, igual que `ciiu3`/`ciiu4`. |
| `segundo_apellido` | 38.8% | No toda Persona Natural tiene segundo apellido. |
| `primer_apellido`, `primer_nombre` | 35.4% | Solo aplica a Persona Natural; Persona Jurídica no tiene nombre/apellido. |
| `nit`, `digito_verificacion` | 1.9% | Solo Persona Jurídica tiene NIT con dígito de verificación; Persona Natural se identifica solo con cédula. Se verificó que `nit` **no es un duplicado exacto** de `numero_identificacion` (coinciden en 98.9% de los casos, pero difieren en 30.938 filas) — ambas columnas se conservan. |

El resto de las 36 columnas (incluyendo las 4 usadas en las reglas de rechazo) tiene 0% o prácticamente 0% de vacíos.

In [ ]:
%sql
-- Evidencia Paso 0: % de vacios por columna (solo las columnas con nulos relevantes)
SELECT
  ROUND(100.0 * SUM(CASE WHEN sigla IS NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_sigla,
  ROUND(100.0 * SUM(CASE WHEN nit IS NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_nit,
  ROUND(100.0 * SUM(CASE WHEN cod_ciiu_act_econ_sec IS NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_ciiu_sec,
  ROUND(100.0 * SUM(CASE WHEN ciiu3 IS NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_ciiu3,
  ROUND(100.0 * SUM(CASE WHEN ciiu4 IS NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_ciiu4,
  ROUND(100.0 * SUM(CASE WHEN fecha_vigencia IS NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_fecha_vigencia,
  ROUND(100.0 * SUM(CASE WHEN representante_legal IS NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_representante_legal,
  ROUND(100.0 * SUM(CASE WHEN primer_apellido IS NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_primer_apellido,
  ROUND(100.0 * SUM(CASE WHEN segundo_nombre IS NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_segundo_nombre,
  ROUND(100.0 * SUM(CASE WHEN fecha_cancelacion IS NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_fecha_cancelacion
FROM Datos_Empresas.bronze.DE_Semiestructurado_RegistroMercantil_Api;

---

## Paso 1: Tipado seguro de columnas

In [ ]:
%python
FORMATO_FECHA_ACTUALIZACION = "yyyy/MM/dd HH:mm:ss.SSSSSSSSS"

# IMPORTANTE: se usa try_to_date / try_to_timestamp (no to_date / to_timestamp).
# Con ANSI SQL activado (el modo por defecto en los SQL Warehouse de Databricks),
# to_date lanza una excepción ante una fecha invalida como '99999997' (mes 99),
# en vez de devolver NULL. try_to_date siempre devuelve NULL si no puede parsear,
# tanto si el string no tiene 8 dígitos como si los 8 dígitos no forman una fecha
# de calendario válida (eso se verificó probando esta consulta en vivo: 533 filas
# de RUES tienen 8 dígitos pero un mes/día imposible).
df_tipado = (
    df_bronze
    .withColumn("matricula", F.trim(F.col("matricula")))
    .withColumn("numero_identificacion", F.trim(F.col("numero_identificacion")))
    .withColumn("fecha_matricula", F.expr("try_to_date(fecha_matricula, 'yyyyMMdd')"))
    .withColumn("fecha_renovacion", F.expr("try_to_date(fecha_renovacion, 'yyyyMMdd')"))
    .withColumn(
        "fecha_vigencia",
        F.when(F.col("fecha_vigencia") == "99991231", F.lit(None))
         .otherwise(F.expr("try_to_date(fecha_vigencia, 'yyyyMMdd')")),
    )
    .withColumn("fecha_cancelacion", F.expr("try_to_date(fecha_cancelacion, 'yyyyMMdd')"))
    .withColumn("ultimo_ano_renovado", F.col("ultimo_ano_renovado").cast("int"))
    .withColumn(
        "fecha_actualizacion",
        F.expr(f"try_to_timestamp(fecha_actualizacion, '{FORMATO_FECHA_ACTUALIZACION}')"),
    )
)

df_tipado.select("matricula", "fecha_matricula", "fecha_vigencia", "fecha_actualizacion").show(5, truncate=False)

---

## Paso 1.5: Diagnóstico previo — evidencia detrás de cada regla

Antes de aplicar las reglas de calidad, se corre el query que justifica cada una contra Bronze. Esto es lo que se corrió para decidir las 4 reglas del notebook (puedes volver a ejecutarlas para verlo tú mismo).

In [ ]:
%sql
-- Evidencia Regla 1: campos obligatorios vacios o numero_identificacion centinela
SELECT
  SUM(CASE WHEN TRIM(matricula) IS NULL OR TRIM(matricula) = ''
        OR TRIM(numero_identificacion) IS NULL OR TRIM(numero_identificacion) = '' THEN 1 ELSE 0 END) AS vacios,
  SUM(CASE WHEN numero_identificacion = '0000000000000' THEN 1 ELSE 0 END) AS centinela_ceros
FROM Datos_Empresas.bronze.DE_Semiestructurado_RegistroMercantil_Api;

In [ ]:
%sql
-- Evidencia Regla 2: (codigo_camara, matricula) duplicada.
-- Se conserva solo la copia con mayor fecha_actualizacion por grupo; el resto se rechaza.
WITH dup AS (
  SELECT codigo_camara, matricula, fecha_actualizacion,
         COUNT(*) OVER (PARTITION BY codigo_camara, matricula) AS dup_count,
         ROW_NUMBER() OVER (PARTITION BY codigo_camara, matricula ORDER BY fecha_actualizacion DESC NULLS LAST) AS rn
  FROM Datos_Empresas.bronze.DE_Semiestructurado_RegistroMercantil_Api
)
SELECT
  SUM(CASE WHEN dup_count > 1 THEN 1 ELSE 0 END) AS filas_en_grupos_duplicados,
  SUM(CASE WHEN dup_count > 1 AND rn = 1 THEN 1 ELSE 0 END) AS se_conservan_version_mas_reciente,
  SUM(CASE WHEN dup_count > 1 AND rn > 1 THEN 1 ELSE 0 END) AS se_rechazan_version_antigua
FROM dup;

In [ ]:
%sql
-- Evidencia Regla 3: fecha_matricula con formato/calendario invalido, en el futuro, o anterior a 1900
-- (el piso de 1900 se agrego tras encontrar 7 filas con el centinela '00010101'/'10000101' que
-- try_to_date acepta como fecha de calendario valida pero no es una fecha real de matricula)
SELECT
  SUM(CASE WHEN try_to_date(fecha_matricula, 'yyyyMMdd') IS NULL THEN 1 ELSE 0 END) AS formato_o_calendario_invalido,
  SUM(CASE WHEN try_to_date(fecha_matricula, 'yyyyMMdd') > current_date() THEN 1 ELSE 0 END) AS fecha_futura,
  SUM(CASE WHEN try_to_date(fecha_matricula, 'yyyyMMdd') < DATE('1900-01-01') THEN 1 ELSE 0 END) AS fecha_anterior_a_1900
FROM Datos_Empresas.bronze.DE_Semiestructurado_RegistroMercantil_Api;

In [ ]:
hoy = F.current_date()
piso_fecha = F.lit("1900-01-01").cast("date")

w_matricula = Window.partitionBy("codigo_camara", "matricula")
# Orden dentro de cada grupo duplicado: la fila mas reciente (mayor fecha_actualizacion)
# queda con rn = 1. NULLS LAST para que una fecha_actualizacion nula nunca "gane" a una valida.
w_matricula_orden = Window.partitionBy("codigo_camara", "matricula").orderBy(
    F.col("fecha_actualizacion").desc_nulls_last()
)

df_reglas = (
    df_tipado
    .withColumn("_dup_count", F.count("*").over(w_matricula))
    .withColumn("_rank_matricula", F.row_number().over(w_matricula_orden))
    # Regla 1: campos obligatorios vacíos, o numero_identificacion es el centinela de
    # "no capturado" ('0000000000000') que comparten 7.338 empresas no relacionadas
    .withColumn(
        "_r1",
        F.when(
            (F.col("matricula").isNull()) | (F.col("matricula") == "")
            | (F.col("numero_identificacion").isNull()) | (F.col("numero_identificacion") == "")
            | (F.col("numero_identificacion") == "0000000000000"),
            F.lit("campo_obligatorio_vacio"),
        ),
    )
    # Regla 2: (codigo_camara, matricula) duplicada -> se conserva solo la version mas
    # reciente (rn = 1) del grupo; las demas copias (rn > 1) se rechazan como version antigua.
    .withColumn(
        "_r2",
        F.when(
            (F.col("_dup_count") > 1) & (F.col("_rank_matricula") > 1),
            F.lit("matricula_duplicada_version_antigua"),
        ),
    )
    # Regla 3: fecha_matricula con formato/calendario inválido (quedó NULL tras el try_to_date
    # del Paso 1), en el futuro, o anterior a 1900 (centinela de dato faltante que try_to_date
    # acepta como fecha de calendario valida, ej. '00010101' o '10000101')
    .withColumn(
        "_r3",
        F.when(F.col("fecha_matricula").isNull(), F.lit("fecha_matricula_formato_invalido"))
         .when(F.col("fecha_matricula") > hoy, F.lit("fecha_matricula_futura"))
         .when(F.col("fecha_matricula") < piso_fecha, F.lit("fecha_matricula_formato_invalido")),
    )
    # Regla 4: cod_ciiu_act_econ_pri vacío (verificado contra Bronze: 31 filas, 100% CANCELADA)
    .withColumn(
        "_r4",
        F.when(
            (F.col("cod_ciiu_act_econ_pri").isNull()) | (F.trim(F.col("cod_ciiu_act_econ_pri")) == ""),
            F.lit("cod_ciiu_vacio"),
        ),
    )
    .withColumn("_motivo_rechazo", F.concat_ws("; ", F.col("_r1"), F.col("_r2"), F.col("_r3"), F.col("_r4")))
    .withColumn(
        "_motivo_rechazo",
        F.when(F.col("_motivo_rechazo") == "", F.lit(None)).otherwise(F.col("_motivo_rechazo")),
    )
)

---

## Paso 2: Aplicar las reglas de calidad y calcular `_motivo_rechazo`

In [ ]:
hoy = F.current_date()

w_matricula = Window.partitionBy("codigo_camara", "matricula")
# Orden dentro de cada grupo duplicado: la fila mas reciente (mayor fecha_actualizacion)
# queda con rn = 1. NULLS LAST para que una fecha_actualizacion nula nunca "gane" a una valida.
w_matricula_orden = Window.partitionBy("codigo_camara", "matricula").orderBy(
    F.col("fecha_actualizacion").desc_nulls_last()
)

df_reglas = (
    df_tipado
    .withColumn("_dup_count", F.count("*").over(w_matricula))
    .withColumn("_rank_matricula", F.row_number().over(w_matricula_orden))
    # Regla 1: campos obligatorios vacíos
    .withColumn(
        "_r1",
        F.when(
            (F.col("matricula").isNull()) | (F.col("matricula") == "")
            | (F.col("numero_identificacion").isNull()) | (F.col("numero_identificacion") == ""),
            F.lit("campo_obligatorio_vacio"),
        ),
    )
    # Regla 2: (codigo_camara, matricula) duplicada -> se conserva solo la version mas
    # reciente (rn = 1) del grupo; las demas copias (rn > 1) se rechazan como version antigua.
    .withColumn(
        "_r2",
        F.when(
            (F.col("_dup_count") > 1) & (F.col("_rank_matricula") > 1),
            F.lit("matricula_duplicada_version_antigua"),
        ),
    )
    # Regla 3: fecha_matricula con formato/calendario inválido (quedó NULL tras el try_to_date
    # del Paso 1) o en el futuro respecto a hoy
    .withColumn(
        "_r3",
        F.when(F.col("fecha_matricula").isNull(), F.lit("fecha_matricula_formato_invalido"))
         .when(F.col("fecha_matricula") > hoy, F.lit("fecha_matricula_futura")),
    )
    # Regla 4: cod_ciiu_act_econ_pri vacío (verificado contra Bronze: 31 filas, 100% CANCELADA)
    .withColumn(
        "_r4",
        F.when(
            (F.col("cod_ciiu_act_econ_pri").isNull()) | (F.trim(F.col("cod_ciiu_act_econ_pri")) == ""),
            F.lit("cod_ciiu_vacio"),
        ),
    )
    .withColumn("_motivo_rechazo", F.concat_ws("; ", F.col("_r1"), F.col("_r2"), F.col("_r3"), F.col("_r4")))
    .withColumn(
        "_motivo_rechazo",
        F.when(F.col("_motivo_rechazo") == "", F.lit(None)).otherwise(F.col("_motivo_rechazo")),
    )
)

---

## Paso 3: Separar aceptados (Silver) de rechazados (DQL)

In [ ]:
columnas_auxiliares = ["_dup_count", "_rank_matricula", "_r1", "_r2", "_r3", "_r4"]

df_silver = (
    df_reglas.filter(F.col("_motivo_rechazo").isNull())
    .drop(*columnas_auxiliares, "_motivo_rechazo", "_ingested_at", "_source")
    .withColumn("_processed_at", F.current_timestamp())
)

df_dql = (
    df_reglas.filter(F.col("_motivo_rechazo").isNotNull())
    .drop(*columnas_auxiliares)
    .withColumn("_fecha_rechazo", F.current_timestamp())
)

total = df_bronze.count()
aceptados = df_silver.count()
rechazados = df_dql.count()
print(f"Bronze: {total:,} | Silver (aceptados): {aceptados:,} | DQL (rechazados): {rechazados:,}")
assert total == aceptados + rechazados, "Se perdieron o duplicaron filas en la separación Silver/DQL"

---

## Paso 4: Persistir (idempotente: `overwrite` recalcula la tabla completa en cada corrida)

In [ ]:
%python
df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLA_SILVER)
df_dql.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLA_DQL)

print(f"Escrito: {TABLA_SILVER}")
print(f"Escrito: {TABLA_DQL}")

---

## Paso 5: Validación

In [ ]:
%sql
SELECT COUNT(*) AS total_silver FROM Datos_Empresas.silver.empresas_rues;


In [ ]:
%sql

SELECT COUNT(*) AS total_dql FROM Datos_Empresas.dql.rechazados_rues;

In [ ]:
%sql
-- Distribución de rechazos por motivo (un registro puede aparecer en más de un motivo compuesto)
SELECT _motivo_rechazo, COUNT(*) AS cantidad
FROM Datos_Empresas.dql.rechazados_rues
GROUP BY _motivo_rechazo
ORDER BY cantidad DESC
LIMIT 20;

In [ ]:
%sql
DESCRIBE EXTENDED Datos_Empresas.silver.empresas_rues;

In [ ]:
%sql
SELECT * FROM Datos_Empresas.silver.empresas_rues LIMIT 10;

---

## Siguiente paso

Continuar con `03_Silver_SECOP.ipynb`, `04_Silver_CIIU.ipynb` y `05_Silver_TRM.ipynb` (son independientes entre sí), y luego `06_Gold_Empresas_Contratos.ipynb`.